# Fine-tune the Smart Irrigation Advisor disease model on PlantDoc

Goal: take the pre-trained PlantVillage EfficientNet (lab photos) and adapt it to **real field photos** by fine-tuning on PlantDoc.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free)
2. Runtime → Run all
3. ~20–40 minutes later, the last cell downloads `model.tflite` to your machine.
4. Replace `backend/app/ml/model.tflite` with the downloaded file. Keep `MODEL_PREPROCESS=raw` in `backend/.env` (the new model still has Rescaling baked in).

**What this notebook does:**
- Downloads the base EfficientNet from HuggingFace (`Nefflymicn/PlantVillage-plant-disease-detection`)
- Clones the PlantDoc dataset and remaps its class names to the 38 PlantVillage keys our backend expects
- Fine-tunes in two stages (head only → top layers unfrozen)
- Evaluates honest top-1 / top-5 accuracy on the PlantDoc test split
- Exports to TFLite and downloads it

## 1. Verify GPU and install deps

In [ ]:
!nvidia-smi | head -20
import tensorflow as tf
print('TF', tf.__version__, 'GPUs', tf.config.list_physical_devices('GPU'))

In [ ]:
!pip install -q huggingface_hub pillow

## 2. Download the base model from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download
import tensorflow as tf
from tensorflow.keras.layers import BatchNormalization

# The HF model was saved with Keras 2; Colab runs Keras 3. Strip the three
# Keras-2-only kwargs so deserialization works. renorm=False was the default,
# so behavior is unchanged.
#
# Guard: only patch once. Re-running this cell would otherwise wrap the
# previous wrapper and cause infinite recursion at load time.
if not getattr(BatchNormalization.__init__, '_pv_kwargs_patched', False):
    _orig_bn_init = BatchNormalization.__init__
    def _patched_bn_init(self, *args, **kwargs):
        for k in ('renorm', 'renorm_clipping', 'renorm_momentum'):
            kwargs.pop(k, None)
        return _orig_bn_init(self, *args, **kwargs)
    _patched_bn_init._pv_kwargs_patched = True
    BatchNormalization.__init__ = _patched_bn_init
    print('BatchNormalization patched.')
else:
    print('BatchNormalization already patched, skipping.')

base_path = hf_hub_download(
    repo_id='Nefflymicn/PlantVillage-plant-disease-detection',
    filename='plant_disease_efficientnet.keras',
)
print('Base model:', base_path)

base_model = tf.keras.models.load_model(base_path, safe_mode=False)
print('Loaded.  Input:', base_model.input_shape, 'Output:', base_model.output_shape)
base_model.summary(line_length=110)

## 3. Clone PlantDoc and remap class names to the 38 PlantVillage keys

In [ ]:
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /content/plantdoc_raw
!ls /content/plantdoc_raw

In [ ]:
# PlantDoc folder name → PlantVillage class key.
# PlantDoc only covers ~28 of the 38 PlantVillage classes; the rest stay unchanged in the model.
PLANTDOC_TO_PLANTVILLAGE = {
    'Apple Scab Leaf':                              'Apple___Apple_scab',
    'Apple leaf':                                   'Apple___healthy',
    'Apple rust leaf':                              'Apple___Cedar_apple_rust',
    'Bell_pepper leaf':                             'Pepper,_bell___healthy',
    'Bell_pepper leaf spot':                        'Pepper,_bell___Bacterial_spot',
    'Blueberry leaf':                               'Blueberry___healthy',
    'Cherry leaf':                                  'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot':                          'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight':                             'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf':                               'Corn_(maize)___Common_rust_',
    'Peach leaf':                                   'Peach___healthy',
    'Potato leaf':                                  'Potato___healthy',
    'Potato leaf early blight':                     'Potato___Early_blight',
    'Potato leaf late blight':                      'Potato___Late_blight',
    'Raspberry leaf':                               'Raspberry___healthy',
    'Soyabean leaf':                                'Soybean___healthy',
    'Squash Powdery mildew leaf':                   'Squash___Powdery_mildew',
    'Strawberry leaf':                              'Strawberry___healthy',
    'Tomato Early blight leaf':                     'Tomato___Early_blight',
    'Tomato Septoria leaf spot':                    'Tomato___Septoria_leaf_spot',
    'Tomato leaf':                                  'Tomato___healthy',
    'Tomato leaf bacterial spot':                   'Tomato___Bacterial_spot',
    'Tomato leaf late blight':                      'Tomato___Late_blight',
    'Tomato leaf mosaic virus':                     'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus':                     'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf':                             'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf':         'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape leaf':                                   'Grape___healthy',
    'grape leaf black rot':                         'Grape___Black_rot',
}

# The 38 classes our backend already knows about (must match backend/app/ml/class_labels.json)
CLASS_LABELS = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy',
    'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy',
    'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot', 'Peach___healthy',
    'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy',
    'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy',
    'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch', 'Strawberry___healthy',
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy',
]
assert len(CLASS_LABELS) == 38
print('Mapping covers', len(set(PLANTDOC_TO_PLANTVILLAGE.values())), 'of 38 classes')

In [ ]:
import os, shutil
from pathlib import Path

RAW = Path('/content/plantdoc_raw')
OUT = Path('/content/plantdoc')
if OUT.exists():
    shutil.rmtree(OUT)

def find_split(name):
    for candidate in [RAW / name, RAW / name.lower(), RAW / name.upper()]:
        if candidate.is_dir():
            return candidate
    return None

for split_name in ('train', 'test'):
    src = find_split(split_name)
    if src is None:
        print(f'No {split_name!r} split found, skipping'); continue
    n_copied = 0
    for plant_dir in src.iterdir():
        if not plant_dir.is_dir():
            continue
        pv_key = PLANTDOC_TO_PLANTVILLAGE.get(plant_dir.name)
        if not pv_key:
            continue   # PlantDoc class with no PlantVillage equivalent → skip
        dest = OUT / split_name / pv_key
        dest.mkdir(parents=True, exist_ok=True)
        for img in plant_dir.iterdir():
            if img.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                shutil.copy2(img, dest / img.name)
                n_copied += 1
    print(f'{split_name}: {n_copied} images across {len(list((OUT / split_name).iterdir()))} classes')

# Sanity dump
for split in ('train', 'test'):
    p = OUT / split
    if p.is_dir():
        print(f'\n{split}:')
        for sub in sorted(p.iterdir()):
            print(f'  {sub.name:55s} {len(list(sub.iterdir()))}')

## 4. Build the training datasets

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path

IMG = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.15),
], name='augment')

# Each split discovers its own classes — train and test don't necessarily have
# the same subset (e.g. Tomato___Spider_mites is in train but not in test).
# Labels are always remapped to the full 38-class index space, so the model's
# softmax head aligns regardless of which subset is present.
def load(split, do_augment):
    split_dir = Path(f'/content/plantdoc/{split}')
    split_classes = sorted([
        d.name for d in split_dir.iterdir()
        if d.is_dir() and d.name in CLASS_LABELS
    ])
    print(f'{split}: {len(split_classes)} classes')
    split_to_full = tf.constant(
        [CLASS_LABELS.index(c) for c in split_classes], dtype=tf.int32
    )
    ds = tf.keras.utils.image_dataset_from_directory(
        str(split_dir),
        image_size=(IMG, IMG),
        batch_size=BATCH,
        label_mode='int',
        class_names=split_classes,
        shuffle=(split == 'train'),
        seed=42,
    )
    ds = ds.map(lambda x, y: (x, tf.gather(split_to_full, y)),
                num_parallel_calls=AUTOTUNE)
    if do_augment:
        ds = ds.map(lambda x, y: (augment(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = load('train', do_augment=True)
test_ds  = load('test',  do_augment=False)

## 5. Baseline accuracy (before fine-tuning)

This is your starting line — anything below this after fine-tuning means we broke the model.

In [ ]:
import numpy as np

# We re-wrap the base model so we can call .evaluate() — it doesn't already have a loss attached.
wrapped = tf.keras.Sequential([base_model])
wrapped.compile(
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5')],
)
print('Baseline (PlantDoc test, no fine-tune):')
baseline_metrics = wrapped.evaluate(test_ds, return_dict=True, verbose=2)
print(baseline_metrics)

## 6. Stage 1 — fine-tune just the classification head

The base model already has the correct 38-class softmax head. We freeze the
entire feature extractor and only let the head's weights adapt — fast and stable.

In [ ]:
from tensorflow.keras import optimizers, callbacks

# Keras 3 quirk: setting model.trainable=False overrides per-layer flags, even
# if you flip layers back to True afterwards. So we go the other way:
# turn the master switch ON, then freeze the layers we want frozen.
base_model.trainable = True
for layer in base_model.layers[:-5]:
    layer.trainable = False
# BatchNorm stays frozen — small batches make trainable BN unstable.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
model = base_model

trainable = sum(int(tf.size(w)) for w in model.trainable_weights)
total = sum(int(tf.size(w)) for w in model.weights)
print(f'Trainable params: {trainable:,} / {total:,}')
print('Top 5 layers:')
for l in base_model.layers[-5:]:
    n = sum(int(tf.size(w)) for w in l.weights)
    print(f'  {l.name:30s} {l.__class__.__name__:25s} params={n:>10,d}  trainable={l.trainable}')
if trainable == 0:
    raise RuntimeError('Still 0 trainable params — inspect base_model.summary() and pick layers by name.')

model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5')],
)

ckpt = callbacks.ModelCheckpoint(
    '/content/best.weights.h5',
    save_best_only=True, save_weights_only=True,
    monitor='val_accuracy', mode='max',
)
early = callbacks.EarlyStopping(
    patience=4, restore_best_weights=True,
    monitor='val_accuracy', mode='max',
)

h1 = model.fit(train_ds, validation_data=test_ds, epochs=8,
               callbacks=[ckpt, early], verbose=2)

## 7. Stage 2 — unfreeze top layers and fine-tune at small LR

In [ ]:
# Find the wrapped feature extractor — it's a nested model (Functional/Sequential)
# containing the actual conv layers. base_model.layers is just a thin wrapper.
backbone = None
for layer in base_model.layers:
    if hasattr(layer, 'layers') and len(layer.layers) > 20:
        backbone = layer
        break
if backbone is None:
    raise RuntimeError('Could not find backbone submodel. Inspect base_model.layers.')
print(f'Backbone: {backbone.name} with {len(backbone.layers)} sublayers')

# Unfreeze top N conv layers of the backbone + keep the dense head trainable.
# BatchNorm stays in inference mode (small batches break trainable BN).
UNFREEZE = 40
base_model.trainable = True
backbone.trainable = True
for layer in backbone.layers[:-UNFREEZE]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable = sum(int(tf.size(w)) for w in model.trainable_weights)
total = sum(int(tf.size(w)) for w in model.weights)
print(f'Trainable params: {trainable:,} / {total:,}')
if trainable <= 796_966:
    raise RuntimeError('Stage 2 should have MORE trainable params than Stage 1.')

model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5')],
)
h2 = model.fit(train_ds, validation_data=test_ds, epochs=20,
               callbacks=[ckpt, early], verbose=2)

## 8. Final evaluation (with test-time augmentation)

In [ ]:
import numpy as np

model.compile(
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5')],
)

print('Plain evaluation:')
plain = model.evaluate(test_ds, return_dict=True, verbose=2)

print('\nTest-time augmentation (original + horizontal flip):')
top1 = top5 = total = 0
for xb, yb in test_ds:
    p1 = model.predict(xb, verbose=0)
    p2 = model.predict(tf.image.flip_left_right(xb), verbose=0)
    probs = (p1 + p2) / 2.0
    pred  = np.argmax(probs, axis=1)
    y     = yb.numpy()
    top1 += int((pred == y).sum())
    top5 += int(sum(y[i] in np.argsort(probs[i])[-5:] for i in range(len(y))))
    total += len(y)
print(f'  TTA top-1: {top1/total:.2%}  top-5: {top5/total:.2%}  (n={total})')
print(f'\nSummary vs baseline:')
print(f'  baseline top-1: {baseline_metrics["accuracy"]:.2%}')
print(f'  fine-tuned:     {plain["accuracy"]:.2%}')
print(f'  + TTA:          {top1/total:.2%}')

## 9. Export to TFLite

In [ ]:
import os
os.makedirs('/content/out', exist_ok=True)
saved_dir = '/content/out/saved_model'
model.export(saved_dir)
print('SavedModel written to', saved_dir)

converter = tf.lite.TFLiteConverter.from_saved_model(saved_dir)
# Optional: dynamic-range INT8 quantization (smaller file, ~same accuracy)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

out_tflite = '/content/out/model.tflite'
with open(out_tflite, 'wb') as f:
    f.write(tflite_bytes)
size_mb = os.path.getsize(out_tflite) / (1024 * 1024)
print(f'TFLite: {out_tflite}  ({size_mb:.1f} MB)')

## 10. Verify the TFLite model loads + sanity-check its outputs

In [ ]:
import numpy as np
from PIL import Image

interp = tf.lite.Interpreter(model_path=out_tflite)
interp.allocate_tensors()
ind = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print('Input :', ind['shape'], ind['dtype'])
print('Output:', out['shape'], out['dtype'])

# Pick one real test image and run it end-to-end
from pathlib import Path
sample = next(iter(Path('/content/plantdoc/test').rglob('*.jpg')))
img = Image.open(sample).convert('RGB')
w, h = img.size
side = min(w, h)
img = img.crop(((w-side)//2, (h-side)//2, (w+side)//2, (h+side)//2)).resize((224, 224))
arr = np.asarray(img, dtype=np.float32)[None, ...]   # raw [0, 255] — Rescaling is baked in

interp.set_tensor(ind['index'], arr)
interp.invoke()
probs = interp.get_tensor(out['index'])[0]
idx = int(np.argmax(probs))
print(f'\nSample: {sample}')
print(f'True : {sample.parent.name}')
print(f'Pred : {CLASS_LABELS[idx]}  (conf={probs[idx]:.3f})')
print(f'Sum of probs: {probs.sum():.4f}  (should be ~1.0)')

## 11. Download the new model.tflite to your machine

In [ ]:
from google.colab import files
files.download('/content/out/model.tflite')

## What to do next on your local machine

```bash
# 1. Replace the model file
cp ~/Downloads/model.tflite backend/app/ml/model.tflite

# 2. Keep MODEL_PREPROCESS=raw in backend/.env (Rescaling is still baked in)

# 3. Restart backend
cd backend && .venv/bin/uvicorn app.main:app --reload

# 4. (Optional) Run the benchmark to verify field accuracy on your own photos
python ../ml/evaluate_model.py \
    --model app/ml/model.tflite \
    --labels app/ml/class_labels.json \
    --images-dir /path/to/your/field_photos \
    --compare-preprocess --tta
```